# Task 4.9 — Single Strain HAI Vic B/Austria/1359417/2021 (D365 Durability)

**4.9 Predict antibody durability - Vic B/Austria/1359417/2021 (D365)**
* Training Data: Demographics + Day 0 + Day 7 innate + **Day 28**
* Assay: HAI / Measure: Post-vac HAI / Metric: Spearman correlation
* Full description: Post-vac HAI of the B Victoria vaccine strain at Day 365

---

## Design notes

**y-values:** log2-transformed. Since Spearman only cares about ranking, no inverse transform is needed for evaluation. Metrics (RMSE, MAE, Spearman) are all in log2 space; CSVs use `np.exp2` to output raw titer scale values.

**Spearman correlation:** ranks predictions and truth; rewards monotonic agreement regardless of scale. Robust to outliers. Score: 1.0 = perfect, 0.0 = no signal, -1.0 = reversed.

**5-fold cross-validation:** each participant's prediction is made by a model that never saw them during training.

In [ ]:
STRAIN = 'Vic B/Austria/1359417/2021'
TARGET_COL = f'HAI_{STRAIN}_d365'
AUTO_ML_MAX_RUNTIME_SECONDS = 1200

In [ ]:
PARQUET_PATH = '../merged_data/combined.parquet'
CHALLENGE_DATA_PATH = '../cleaned_data'
SUBMISSION_PATH = '../automl_submission'

In [ ]:
import io
import warnings
from contextlib import redirect_stderr, redirect_stdout

import h2o
import numpy as np
import pandas as pd
from h2o.automl import H2OAutoML
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=UserWarning, module="h2o")
h2o.init()

In [ ]:
data = h2o.import_file(PARQUET_PATH)
print(f'Training data shape: {data.shape}')

challenge_participants = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_participants_cleaned.csv')
challenge_hai = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_hai_cleaned.csv')
challenge_data = challenge_hai.merge(challenge_participants, on='participant_id', how='inner')
print(f'Challenge shape: {challenge_data.shape}')

---
## AutoML Training

In [ ]:
# Features: d0 + d7 + d28 allowed for durability task — exclude only d365 targets and participant_id
x = [c for c in data.columns
     if not c.endswith('_d365')
     and c != 'participant_id']
y = TARGET_COL

train = data[data[y].isna() == 0]
print(f'Training samples: {train.nrows}  |  Features: {len(x)}')

aml = H2OAutoML(max_models=10, seed=1, nfolds=5,
                keep_cross_validation_predictions=True,
                max_runtime_secs=AUTO_ML_MAX_RUNTIME_SECONDS)

_buf = io.StringIO()
with redirect_stdout(_buf), redirect_stderr(_buf):
    aml.train(x=x, y=y, training_frame=train)
print('Training complete.')

In [ ]:
lb = aml.leaderboard
print(lb.head(rows=lb.nrows))

In [ ]:
cv_preds = aml.leader.cross_validation_holdout_predictions().as_data_frame()['predict']
actuals = train[y].as_data_frame()[y]
rho, pval = spearmanr(actuals, cv_preds)
print(f'Task 4.9 — Spearman (5-fold CV): {rho:.3f}  (p={pval:.4f})')

In [ ]:
print(f'Leader model: {aml.leader.model_id}')
varimp = aml.leader.varimp(use_pandas=True)
display(varimp.head(20))
aml.leader.varimp_plot(num_of_features=20)

In [ ]:
challenge_hf = h2o.H2OFrame(challenge_data)
y_pred = aml.leader.predict(challenge_hf).as_data_frame()['predict']

results = pd.DataFrame({
    'Participant_ID': challenge_data['participant_id'].values,
    'Task_4.9': np.exp2(y_pred),
})
results.to_csv(f'{SUBMISSION_PATH}/task_4_9.csv', index=False)
results

In [ ]:
h2o.cluster().shutdown()

---
## Conclusion

- **Leader model:** (fill after run)
- **CV Spearman:** (fill after run)

**Target:** log2 HAI titer for Vic B/Austria/1359417/2021 at D365 (durability). The D28 titer for the same strain is expected to be a top predictor; durability typically reflects a broader profile across strains and timepoints than the acute response.

Submission saved to `automl_submission/task_4_9.csv` (raw titer scale via `np.exp2`).